# Importing Libraries

In [1]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from mlflow.models.signature import infer_signature
import xgboost as xgb
import lightgbm as lgb
from itertools import product


# Model Configurations


In [2]:
model_configs = [
    {
        "name": "RandomForest",
        "class": RandomForestClassifier,
        "param_grid": {
            "n_estimators": [100, 200],
            "max_depth": [10, None],
            "class_weight": ["balanced"]
        }
    },
    {
        "name": "LogisticRegression",
        "class": LogisticRegression,
        "param_grid": {
            "C": [0.1, 1.0],
            "penalty": ["l2"],
            "class_weight": ["balanced"],
            "solver": ["liblinear"],
            "max_iter": [500]
        }
    },
    {
        "name": "XGBoost",
        "class": xgb.XGBClassifier,
        "param_grid": {
            "n_estimators": [100],
            "max_depth": [5, 10],
            "learning_rate": [0.1],
            "eval_metric": ["logloss"],
            "use_label_encoder": [False]
        }
    },
    {
        "name": "LightGBM",
        "class": lgb.LGBMClassifier,
        "param_grid": {
            "n_estimators": [100],
            "max_depth": [5, 10],
            "learning_rate": [0.1],
            "boosting_type": ["gbdt"]
        }
    }
]


# Dataset Preparation

In [3]:
def prepare_data(path="WA_Fn-UseC_-Telco-Customer-Churn.csv"):
    df = pd.read_csv(path)
    df = df.dropna()
    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
    X = pd.get_dummies(df.drop(columns=['customerID', 'Churn']), drop_first=True)
    y = df['Churn']
    return train_test_split(X, y, test_size=0.2, random_state=42)


# Function to evaluate and log the models

In [4]:
def evaluate_and_log(model, name, X_train, X_test, y_train, y_test):
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        mlflow.log_params(model.get_params())
        mlflow.log_param("model_name", name)
        mlflow.log_metrics({
            "accuracy": acc,
            "precision": prec,
            "recall": recall,
            "f1_score": f1
        })
        mlflow.set_tag("type", "ensemble_comparison")
        signature = infer_signature(X_train, y_pred)
        mlflow.sklearn.log_model(model, "model", signature=signature, input_example=X_train.iloc[:1])

        print(f"[{name}] acc: {acc:.4f} | prec: {prec:.4f} | recall: {recall:.4f} | f1: {f1:.4f}")


In [5]:
mlflow.set_experiment("TelcoChurn_MultiModel_ParamSearch")
X_train, X_test, y_train, y_test = prepare_data()

for config in model_configs:
    keys, values = zip(*config["param_grid"].items())
    for combination in product(*values):
        params = dict(zip(keys, combination))
        model = config["class"](**params)
        run_name = f"{config['name']}_" + "_".join(f"{k}={v}" for k, v in params.items())
        evaluate_and_log(model, run_name, X_train, X_test, y_train, y_test)


e:\Mlops\mlflow_env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


[RandomForest_n_estimators=100_max_depth=10_class_weight=balanced] acc: 0.7090 | prec: 0.4729 | recall: 0.8660 | f1: 0.6117


e:\Mlops\mlflow_env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


[RandomForest_n_estimators=100_max_depth=None_class_weight=balanced] acc: 0.7949 | prec: 0.6556 | recall: 0.4745 | f1: 0.5505


e:\Mlops\mlflow_env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


...

[XGBoost_n_estimators=100_max_depth=10_learning_rate=0.1_eval_metric=logloss_use_label_encoder=False] acc: 0.7850 | prec: 0.6199 | recall: 0.4853 | f1: 0.5444
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1496, number of negative: 4138
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000776 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 382
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265531 -> initscore=-1.017418
[LightGBM] [Info] Start training from score -1.017418
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

e:\Mlops\mlflow_env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


[LightGBM_n_estimators=100_max_depth=5_learning_rate=0.1_boosting_type=gbdt] acc: 0.8077 | prec: 0.6656 | recall: 0.5496 | f1: 0.6021
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1496, number of negative: 4138
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000207 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 382
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265531 -> initscore=-1.017418
[LightGBM] [Info] Start training from score -1.017418
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightG

e:\Mlops\mlflow_env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


[LightGBM_n_estimators=100_max_depth=10_learning_rate=0.1_boosting_type=gbdt] acc: 0.8020 | prec: 0.6556 | recall: 0.5308 | f1: 0.5867


# Import best 3 model


In [12]:
from mlflow.tracking import MlflowClient
import mlflow.sklearn

client = MlflowClient()
experiment = client.get_experiment_by_name("TelcoChurn_MultiModel_ParamSearch")
runs = client.search_runs(experiment.experiment_id, order_by=["metrics.f1_score DESC"], max_results=3)


# Save and print best 3 model

In [13]:
models = []
for run in runs:
    model_uri = f"runs:/{run.info.run_id}/model"
    model = mlflow.sklearn.load_model(model_uri)
    models.append((run.data.params['model_name'], model))
print(f"Top 3 models: {[name for name, _ in models]}")

Top 3 models: ['LogisticRegression_C=1.0_penalty=l2_class_weight=balanced_solver=liblinear_max_iter=500', 'LogisticRegression_C=0.1_penalty=l2_class_weight=balanced_solver=liblinear_max_iter=500', 'RandomForest_n_estimators=100_max_depth=10_class_weight=balanced']


# Voting Classifer 

In [20]:
from sklearn.ensemble import VotingClassifier

voting_model = VotingClassifier(estimators=models, voting='soft')  # 'soft' için predict_proba lazım
voting_model.fit(X_train, y_train)
import json 
with open("feature_names.json", "w") as f:
    json.dump(list(X_train.columns), f)

y_pred = voting_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"[VotingClassifier] acc: {acc:.4f} | prec: {prec:.4f} | recall: {recall:.4f} | f1: {f1:.4f}")


[VotingClassifier] acc: 0.7693 | prec: 0.5430 | recall: 0.8123 | f1: 0.6509


# Log the Voting Classfier to the MLFlow


In [21]:
from mlflow.models.signature import infer_signature
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import mlflow.sklearn

with mlflow.start_run(run_name="VotingClassifier_ensemble"):
    voting_model.fit(X_train, y_train)
    y_pred = voting_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_param("model_name", "VotingClassifier")
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.set_tag("type", "ensemble_final")
    mlflow.register_model(
    model_uri="runs:/{}/model".format(mlflow.active_run().info.run_id),
    name="TelcoChurnVotingEnsemble"
)


    signature = infer_signature(X_train, y_pred)
    mlflow.sklearn.log_model(voting_model, "model", signature=signature, input_example=X_train.iloc[:1])


Registered model 'TelcoChurnVotingEnsemble' already exists. Creating a new version of this model...
Created version '2' of model 'TelcoChurnVotingEnsemble'.
e:\Mlops\mlflow_env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


# Model Staging


In [22]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Modeli ve versiyonu staging'e geçir
client.transition_model_version_stage(
    name="TelcoChurnVotingEnsemble",
    version=1,  # ya da doğru versiyon neyse
    stage="Staging"
)


C:\Users\ahmet\AppData\Local\Temp\ipykernel_21772\2641837078.py:6: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1746100560267, current_stage='Staging', description=None, last_updated_timestamp=1746102835935, name='TelcoChurnVotingEnsemble', run_id='2af02da918084bc98a00b7806c19ea86', run_link=None, source='file:///E:/Mlops/mlruns/694330000616348606/2af02da918084bc98a00b7806c19ea86/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=1>

# Model Deployment


In [ ]:
# Using Streamlit to create a simple web app check app.py
